# 04 · Profit levers (sized on the last 12 months, Jun 2024–May 2025)

Each lever is sized transparently from actual flights, holding everything else equal:
1. **Fill seats** on "Fill seats" routes (good RASK, low load factor) up to the network load factor, at each route's own fare.
2. **Lift yield** on "Raise yield" routes (full planes, low RASK) by 5% in fares, assuming the Nov 2024 evidence holds (demand did not fall).
3. **Re-deploy capacity** from the weakest "Review" routes (bottom quartile RASK) to Star routes at the Star routes' average RASK.
4. **International load factor** from its current level to halfway towards the domestic level.

In [ ]:
import sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from src.metrics import load_clean, headline, DATA_DIR, OUTPUT_DIR, DATA_END
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
NAVY, VIOLET, LILAC, ORANGE, SLATE, GREEN = "#030734", "#682EFF", "#CD83FF", "#F05100", "#62748E", "#009588"
sns.set_theme(style="whitegrid", palette=[VIOLET, LILAC, NAVY, ORANGE, SLATE, GREEN])
FIG = OUTPUT_DIR / "figures"; FIG.mkdir(parents=True, exist_ok=True); DATA_DIR.mkdir(parents=True, exist_ok=True)
Cr = lambda v: f"₹{v/1e7:,.2f} Cr"; Lk = lambda v: f"₹{v/1e5:,.1f} L"
findings = []
def note(q, topic, text):
    findings.append(dict(q=q, topic=topic, finding=text)); print(f"[{q}] {topic}: {text}")
def save(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=150, bbox_inches="tight"); plt.show()
def kpis(g):
    return pd.Series({"flights": len(g), "revenue": g.Revenue.sum(), "pax": g.PaxCount.sum(), "seats": g.Capacity.sum(),
                      "ask": g.ask.sum(), "rpk": g.rpk.sum(), "lf": g.PaxCount.sum() / g.Capacity.sum(),
                      "fare": g.Revenue.sum() / g.PaxCount.sum(), "rask": g.Revenue.sum() / g.ask.sum(),
                      "yield_rpk": g.Revenue.sum() / g.rpk.sum(), "stage_km": g.ask.sum() / g.Capacity.sum()})

In [ ]:
f = pd.read_parquet(DATA_DIR / "flights.parquet")
NET = kpis(f)

In [ ]:
L12 = f[f.date > DATA_END - pd.DateOffset(months=12)]
NET12 = kpis(L12)
R = L12.groupby("route").apply(kpis, include_groups=False)
R["quadrant"] = np.select([(R.lf >= NET12.lf) & (R.rask >= NET12.rask), (R.lf < NET12.lf) & (R.rask >= NET12.rask),
                           (R.lf >= NET12.lf) & (R.rask < NET12.rask)], ["Star", "Fill seats", "Raise yield"], "Review route")
fs = R[R.quadrant == "Fill seats"]
lever1 = ((NET12.lf - fs.lf).clip(lower=0) * fs.seats * fs.fare).sum()
ry = R[R.quadrant == "Raise yield"]; lever2 = 0.05 * ry.revenue.sum()
rv = R[R.quadrant == "Review route"]; weak = rv[rv.rask <= rv.rask.quantile(.25)]
# Value re-deployed seat-km at the NETWORK average RASK (conservative: Star-route RASK is inflated by the PNQ-BOM pricing outlier)
lever3 = (weak.ask.sum() * NET12.rask) - weak.revenue.sum()
I = kpis(L12[L12.market == "International"]); D = kpis(L12[L12.market == "Domestic"])
lever4 = ((D.lf - I.lf) / 2) * I.seats * I.fare
LV = pd.DataFrame([
    dict(lever="Fill seats on high-RASK routes", uplift=lever1, basis=f"{len(fs)} routes lifted to network LF {NET12.lf:.1%}", confidence="Medium"),
    dict(lever="5% fare rise on full, low-RASK routes", uplift=lever2, basis=f"{len(ry)} 'Raise yield' routes; Nov 2024 fare rise did not cut demand", confidence="Medium"),
    dict(lever="Re-deploy weakest route capacity", uplift=lever3, basis=f"{len(weak)} bottom-quartile review routes ({weak.flights.sum():,.0f} flights) flown at network-average RASK", confidence="Low-medium"),
    dict(lever="International load factor halfway to domestic", uplift=lever4, basis=f"LF {I.lf:.0%} to {(I.lf + D.lf) / 2:.0%}", confidence="Medium"),
])
LV["uplift_pct_of_revenue"] = LV.uplift / NET12.revenue
LV.to_csv(OUTPUT_DIR / "pbi_levers.csv", index=False); display(LV)
note("Levers", "Sized on Jun 2024-May 2025", " | ".join(f"{r.lever}: +{Cr(r.uplift)} ({r.uplift_pct_of_revenue:.1%})" for _, r in LV.iterrows()) +
     f". Base revenue {Cr(NET12.revenue)}.")
F = pd.DataFrame(findings); F.to_csv(OUTPUT_DIR / "lever_findings.csv", index=False); F

In [ ]:
# ---- facts for the Power BI report text (read by _build/build_report.py) ----
E = json.load(open(OUTPUT_DIR / "eda_numbers.json"))
Y24, Y25 = {k: v["2024"] for k, v in E["Y"].items()}, {k: v["2025"] for k, v in E["Y"].items()}
FA = pd.read_csv(OUTPUT_DIR / "pbi_forecast_2025.csv")
fa = FA[(FA.metric == "Revenue 2025 full year") & FA.scenario.str.startswith("A")].iloc[0]
fb = FA[(FA.metric == "Revenue 2025 full year") & FA.scenario.str.startswith("B")].iloc[0]
Q = pd.DataFrame(E["quadrants"]); Mk = pd.DataFrame(E["market"])
BT = pd.read_csv(OUTPUT_DIR / "forecast_backtest.csv")
pct = lambda a, b: f"{a / b - 1:+.0%}"
facts = {
    "pax_yoy": pct(Y25["pax"], Y24["pax"]), "rev_yoy": pct(Y25["revenue"], Y24["revenue"]),
    "fare_step": f"{E['post']['fare'] / E['pre']['fare'] - 1:.0%}", "lf_yoy_pp": f"{(Y25['lf'] - Y24['lf']) * 100:+.1f}",
    "lf_2025": f"{Y25['lf']:.1%}", "rask_2025": f"Rs {Y25['rask']:.3f}", "rask_yoy": pct(Y25["rask"], Y24["rask"]),
    "rask_status": "on target" if Y25["rask"] >= Y24["rask"] * 1.05 else "below target",
    "f25_a": f"Rs {fa.p50 / 1e7:.1f} Cr", "f25_b": f"Rs {fb.p50 / 1e7:.1f} Cr",
    "bt_err": f"{BT.model.sum() / BT.actual.sum() - 1:+.1%}", "bt_naive": f"{BT.naive_last3.sum() / BT.actual.sum() - 1:+.1%}",
    "findings": [
        ["1. Growth is price-led. ", f"Jan to May 2025 vs 2024: passengers {pct(Y25['pax'], Y24['pax'])}, revenue {pct(Y25['revenue'], Y24['revenue'])}. "
         f"Fares rose {E['post']['fare'] / E['pre']['fare'] - 1:.0%} from Nov 2024 and load factor still went up."],
        ["2. Pune is the network. ", "31% of revenue departs PNQ; 86% of flights touch PNQ, BOM or BLR. 45 of 179 routes make 80% of revenue."],
        ["3. 53 star routes carry 40% of revenue. ", f"48 routes sit below the network on both load factor and RASK (19% of revenue), led by BLR-BOM and DEL-GOX."],
        ["4. RASK is on target. ", f"Rs {Y25['rask']:.3f} per seat-km in 2025, {pct(Y25['rask'], Y24['rask'])} vs last year; yield explains almost 90% of the gain."],
        ["5. International underperforms. ", f"4% of flights, 7% of revenue, but load factor {Mk['lf']['International']:.0%} vs {Mk['lf']['Domestic']:.0%} domestic."],
        ["6. Summer is the soft spot. ", "Jul to Oct 2024 ran at 55.9% load factor and the lowest RASK; there is no weekday pattern."],
        ["7. 2025 outlook: ", f"about Rs {fa.p50 / 1e7:.1f} Cr revenue if fares hold, Rs {fb.p50 / 1e7:.1f} Cr if they settle, vs Rs 40.1 Cr in 2024."],
        ["8. Biggest lever: ", f"move capacity off the weakest routes (+Rs {LV.uplift.iloc[2] / 1e7:.2f} Cr a year), then a 5% fare rise on full, low-RASK routes (+Rs {LV.uplift.iloc[1] / 1e7:.2f} Cr)."],
    ],
    "data_notes": [
        ["Distances corrected. ", "5,660 flights (14%) on 19 routes carried another route's distance (PNQ-BOM as 1,138 km, the Delhi-Mumbai distance). Great-circle distance from airport coordinates replaces them."],
        ["PNQ-BOM pricing. ", "124 km but priced above BLR-BOM (834 km): its fare looks based on the wrong distance, so its RASK (Rs 1.68) is an outlier."],
        ["Scaled figures. ", "No flight exceeds 66.7% load and fares average Rs 293 per passenger: passengers and revenue look scaled. Comparisons hold; absolute values are illustrative."],
        ["Fare outliers flagged. ", "86 flights priced above 3x their route's median fare (0.5% of revenue) are flagged, not removed."],
        ["Repeated flight numbers. ", "930 flights share a number, date and route with another flight but differ in seats, passengers and revenue: kept as separate flights."],
        ["Definitions. ", "ASK = seats x km, RPK = passengers x km, RASK = revenue / ASK. Dates run 1 Jan 2024 to 31 May 2025; year-on-year compares Jan to May."],
    ],
}
json.dump(facts, open(OUTPUT_DIR / "report_facts.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
facts["findings"][:3]